# Instruments

An "instrument" is just a function that runs inside `clm.Sound` and writes samples. pysndlib ships the classic CLM instrument collection in `pysndlib.instruments` (fm_violin, fm_bell, pluck, gong, ...) plus a set of translations in `pysndlib.clm_ins`.

Docs: [CLM instruments](https://ccrma.stanford.edu/software/snd/snd/sndclm.html) -- the instrument chapters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm
import pysndlib.sndlib as sndlib
from pysndlib.instruments import fm_violin, fm_bell, pluck, gong, two_tab

SR = 44100

## Writing your first instrument

An instrument computes its own sample loop. Here is a minimal sine bell:

In [ ]:
def sine_bell(beg, dur, freq, amp):
    start = clm.seconds2samples(beg)
    length = clm.seconds2samples(dur)
    osc = clm.make_oscil(freq)
    ampe = clm.make_env([0, 0, 5, 1, 100, 0], scaler=amp, duration=dur)
    for i in range(start, start + length):
        clm.outa(i, clm.env(ampe) * clm.oscil(osc))

with clm.Sound('first_instrument.wav', channels=1, srate=SR):
    sine_bell(0.0, 0.5, 440.0, 0.3)
    sine_bell(0.5, 0.5, 660.0, 0.3)
    sine_bell(1.0, 1.0, 330.0, 0.3)

In [ ]:
ipd.Audio('first_instrument.wav', rate=SR)

## The classic: fm_violin

`fm_violin` is CLM's canonical instrument -- Chowning's violin from the FM literature (see the [FM notebooks](06_FM_Theory.ipynb)). Its `fm_index` controls brightness:

In [ ]:
import inspect
print(inspect.signature(fm_violin))

In [ ]:
with clm.Sound('violin.wav', channels=1, srate=SR, statistics=True):
    fm_violin(0.0, 2.0, 440.0, 0.4)                 # soft
    fm_violin(2.0, 2.0, 440.0, 0.4, fm_index=4.0)  # bright!

In [ ]:
ipd.Audio('violin.wav', rate=SR)

In [ ]:
vd, _ = clm.file2ndarray('violin.wav')

def spectrum(x, n=4096):
    rl = np.copy(x[:n].astype(float)); im = np.zeros(n)
    clm.mus_fft(rl, im, n, 1)
    clm.mus_rectangular2polar(rl, im)
    return np.abs(rl[:n//2])

f = np.arange(2048) * SR / 4096
plt.figure(figsize=(14, 3))
plt.plot(f, spectrum(vd[0][int(SR*0.4):int(SR*0.4)+4096]), alpha=0.6, label='fm_index=1')
plt.plot(f, spectrum(vd[0][int(SR*2.6):int(SR*2.6)+4096]), alpha=0.6, label='fm_index=4')
plt.xlim(0, 6000); plt.legend(); plt.title('index -> spectral richness')
plt.show()

## A quick tour of the collection

In [ ]:
with clm.Sound('tour.wav', channels=1, srate=SR):
    fm_bell(0.0, 2.0, 440.0, 0.3)      # inharmonic FM bell
    pluck(2.0, 1.5, 220.0, 0.4)         # Karplus-Strong string
    gong(3.5, 2.0, 150.0, 0.4)           # detuned strike
    two_tab(5.5, 1.5, 330.0, 0.3)     # two wavetables

In [ ]:
ipd.Audio('tour.wav', rate=SR)

## clm_ins instruments

`pysndlib.clm_ins` contains translations from clm-ins.scm. `vox` sings phonemes by interpolating formant tables:

In [ ]:
from pysndlib.clm_ins import vox

with clm.Sound('vox.wav', channels=1, srate=SR):
    vox(0.0, 1.0, 170, 0.4,
        [0, 0, 25, 1, 75, 1, 100, 0],           # amplitude envelope
        [0, 0, 5, .5, 10, 0, 100, 1],           # frequency envelope
        .1,                                     # frequency scaler
        [0, 'E', 25, 'AE', 35, 'ER', 65, 'ER', 75, 'I', 100, 'UH'],  # phonemes
        [.8, .15, .05],                         # formant amplitudes
        [.005, .0125, .025],                    # formant indices
        .05, .1)                                # vibrato amount, degree

In [ ]:
ipd.Audio('vox.wav', rate=SR)

## Adding reverb

The reverb instruments (jc_reverb, nrev, jl_reverb) process an intermediate reverb file. Pass the instrument to `Sound` along with its parameters:

In [ ]:
from pysndlib.instruments import jc_reverb

with clm.Sound('reverbed.wav', channels=1, srate=SR,
               reverb=jc_reverb, reverb_data={'decay_time': 2.0},
               reverb_channels=1, revfile='reverbed.rev'):
    fm_bell(0.0, 1.0, 330.0, 0.4)
    pluck(0.5, 1.5, 220.0, 0.4)

In [ ]:
ipd.Audio('reverbed.wav', rate=SR)

## Summary

- Instruments are functions that write samples inside `Sound`
- The `pysndlib.instruments` package is the classic CLM collection, compiled with Cython
- Reverbs are instruments too, wired through `Sound(..., reverb=...)`

Next: [speeding instruments up with Cython](05_Speed_with_Cython.ipynb)